# Heston puis rough Heston — génération directe prix-delta

Exécute la première cellule pour Heston, puis la seconde pour rough Heston. Chaque cellule vérifie/recompile sa cible dans `build`, lance directement l'exécutable obtenu et affiche la progression. Les résultats sont écrits dans les chemins `datasets/` et `catalog/` de la recette ; le notebook refuse de remplacer une sortie déjà présente. Les journaux sont sous `datasets/generation-runs/`. Interrompre une cellule arrête le suivi, pas le générateur ; attends la fin de son PID avant de lancer le suivant.


In [ ]:
from datetime import datetime
from html import escape
import json
import os
from pathlib import Path
import subprocess
import time
from IPython.display import HTML, clear_output, display

ROOT = Path.cwd().resolve()
if not (ROOT / 'tools/datasets/generate_catalog.py').is_file():
    raise RuntimeError('Ouvre le notebook depuis la racine AI_factory.')

BUILD = ROOT / 'build'
TRACKING = ROOT / 'datasets/generation-runs' / datetime.now().strftime('%Y%m%d-%H%M%S-%f')
POLL_SECONDS = 5
PRODUCT_INPUT = ROOT / 'datasets/product/european_option/european_options_01.json'
RECIPES = {
    'generate_heston_european_calls_01_cartesian_price_delta': {
        'input': ROOT / 'datasets/model/equity/markovian/heston/parameters/heston_01.json',
        'output': ROOT / 'datasets/model/equity/markovian/heston/price_delta/european_calls/heston_01__european_calls_01__01_cartesian_price_delta.json',
        'catalog': ROOT / 'catalog/model/equity/markovian/heston/price_delta/european_calls/heston_01__european_calls_01__01_cartesian_price_delta/dataset.yaml',
    },
    'generate_rough_heston_european_calls_01_cartesian_price_delta': {
        'input': ROOT / 'datasets/model/equity/rough/rough_heston/parameters/rough_heston_01.json',
        'output': ROOT / 'datasets/model/equity/rough/rough_heston/price_delta/european_calls/rough_heston_01__european_calls_01__01_cartesian_price_delta.json',
        'catalog': ROOT / 'catalog/model/equity/rough/rough_heston/price_delta/european_calls/rough_heston_01__european_calls_01__01_cartesian_price_delta/dataset.yaml',
    },
}
ACTIVE_PROCESS = None

def duration(seconds):
    if seconds is None:
        return 'indisponible'
    seconds = max(0, int(seconds))
    hours, remainder = divmod(seconds, 3600)
    minutes, seconds = divmod(remainder, 60)
    return f'{hours:02d}:{minutes:02d}:{seconds:02d}'

def run_generator(target):
    global ACTIVE_PROCESS
    if ACTIVE_PROCESS is not None and ACTIVE_PROCESS.poll() is None:
        raise RuntimeError(f'Le générateur PID {ACTIVE_PROCESS.pid} tourne encore.')
    recipe = RECIPES[target]
    existing = [path for path in (recipe['output'], recipe['catalog']) if path.exists()]
    if existing:
        raise FileExistsError(f'Lancement direct refusé : sortie existante {existing}.')
    if not (BUILD / 'CMakeCache.txt').is_file():
        print(f'Configuration CMake : {BUILD}')
        subprocess.run(['cmake', '--preset', 'dev', '-B', str(BUILD)], cwd=ROOT, check=True)
    print(f'Vérification et compilation de {target}...')
    subprocess.run(['cmake', '--build', str(BUILD), '--target', target, '-j2'],
                   cwd=ROOT, check=True)
    binary = BUILD / target
    if not binary.is_file():
        raise FileNotFoundError(f'Exécutable absent après compilation : {binary}')
    total = (json.loads(recipe['input'].read_text())['row_count']
             * json.loads(PRODUCT_INPUT.read_text())['row_count'])
    logs = TRACKING / target
    logs.mkdir(parents=True, exist_ok=False)
    snapshot = logs / 'progress.json'
    journal = logs / 'progress.jsonl'
    stdout = logs / 'stdout.log'
    stderr = logs / 'stderr.log'
    env = {**os.environ,
           'AI_FACTORY_GENERATION_PROGRESS': str(snapshot),
           'AI_FACTORY_GENERATION_PROGRESS_LOG': str(journal)}
    with stdout.open('w') as out, stderr.open('w') as err:
        ACTIVE_PROCESS = subprocess.Popen(
            [str(binary)], cwd=ROOT, env=env, stdout=out, stderr=err,
            start_new_session=True,
        )
    process = ACTIVE_PROCESS
    try:
        while True:
            try:
                progress = json.loads(snapshot.read_text())
            except (FileNotFoundError, json.JSONDecodeError):
                progress = {}
            completed = int(progress.get('completed_prices', 0))
            percent = min(100.0, max(0.0, float(progress.get('percent', 0))))
            rate = float(progress.get('prices_per_second', 0))
            clear_output(wait=True)
            display(HTML(
                f'<h3>{escape(target)}</h3>'
                f'<progress value="{percent:.4f}" max="100" style="width:80%"></progress>'
                f'<p>{completed:,} / {total:,} prix — {percent:.2f} %<br>'
                f'Débit : {rate:.2f} prix/s<br>'
                f'ETA : {duration(progress.get("estimated_seconds_remaining"))}</p>'
            ))
            print(f'PID : {process.pid} | journal : {journal} | stderr : {stderr}')
            returncode = process.poll()
            if returncode is not None:
                break
            time.sleep(POLL_SECONDS)
    except KeyboardInterrupt:
        print(f'Suivi interrompu ; le générateur PID {process.pid} continue. Journal : {journal}')
        return process
    if returncode != 0:
        raise RuntimeError(f'{target} a échoué (code {returncode}). Voir {stderr}.')
    if not recipe['output'].is_file() or not recipe['catalog'].is_file():
        raise RuntimeError(f'{target} a quitté sans écrire ses deux sorties. Voir {stderr}.')
    print(f'Génération terminée. Base : {recipe["output"]}')
    return process

run_generator('generate_heston_european_calls_01_cartesian_price_delta')


In [ ]:
run_generator('generate_rough_heston_european_calls_01_cartesian_price_delta')
